In [ ]:
import torch.nn as nn

class ResidualBlock(nn.Module):
	def __init__(self, in_channels, out_channels, stride=1):
		super(ResidualBlock, self).__init__()
		self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
							   stride=stride, padding=1, bias=False)
		self.bn1 = nn.BatchNorm2d(out_channels)
		self.relu = nn.ReLU(inplace=True)
		self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
							   stride=1, padding=1, bias=False)
		self.bn2 = nn.BatchNorm2d(out_channels)

		# Skip Connection (차원이 변경될 경우 1×1 합성곱으로 조정)
		self.shortcut = nn.Sequential()
		if stride != 1 or in_channels != out_channels:
			self.shortcut = nn.Sequential(
				nn.Conv2d(in_channels, out_channels, kernel_size=1,
						  stride=stride, bias=False),
				nn.BatchNorm2d(out_channels)
			)

	def forward(self, x):
		residual = x
		out = self.conv1(x)
		out = self.bn1(out)
		out = self.relu(out)
		out = self.conv2(out)
		out = self.bn2(out)
		out += self.shortcut(residual)   # Skip Connection
		out = self.relu(out)
		return out

In [7]:
import torch

residual_block = ResidualBlock(64, 64)
test_input = torch.randn(1, 64, 32, 32)
residual_output = residual_block(test_input)

print(test_input.shape)
print(residual_output.shape)

torch.Size([1, 64, 32, 32])
torch.Size([1, 64, 32, 32])


In [14]:
# REsNet
class BasicBlock(nn.Module):
	expansion = 1
	def __init__(self, in_channels, out_channels, stride = 1):
		super(BasicBlock, self).__init__()
		self.conv1 = nn.Conv2d(in_channels,
							   out_channels,
							   kernel_size = 3,
							   stride = stride,
							   padding = 1,
							   bias = False)
		self.bn1 = nn.BatchNorm2d(out_channels)
		self.relu = nn.ReLU(inplace = True)
		self.conv2 = nn.Conv2d(out_channels,
							   out_channels,
							   kernel_size = 3,
							   stride = 1,
							   padding = 1,
							   bias = False)
		self.bn2 = nn.BatchNorm2d(out_channels)
		self.shortcut = nn.Sequential()
		if stride != 1 or in_channels != out_channels * self.expansion:
			self.shortcut = nn.Sequential(
				nn.Conv2d(in_channels,
						  out_channels * self.expansion,
						  kernel_size = 1,
						  stride = stride,
						  bias = False),
				nn.BatchNorm2d(out_channels * self.expansion)
			)

	def forward(self, x):
		residual = x
		out = self.conv1(x)
		out = self.bn1(out)
		out = self.relu(out)
		out = self.conv2(out)
		out = self.bn2(out)
		out += self.shortcut(residual)   # Skip Connection
		out = self.relu(out)
		return out

class ResNet(nn.Module):
	def __init__(self, block, num_blocks, num_classes = 10):
		super(ResNet, self).__init__()
		self.in_channels = 64
		self.conv1 = nn.Conv2d(3,
						 64,
						 kernel_size = 3,
						 stride = 1,
						 padding = 1,
						 bias = False)
		self.bn1 = nn.BatchNorm2d(64)
		self.relu = nn.ReLU(inplace = True)
		self.layer1 = self._make_layer(block,
								 64,
								 num_blocks[0],
								 stride = 1)
		self.layer2 = self._make_layer(block,
								 128,
								 num_blocks[1],
								 stride = 2)
		self.layer3 = self._make_layer(block,
								 256,
								 num_blocks[2],
								 stride = 2)
		self.layer4 = self._make_layer(block,
								 512,
								 num_blocks[3],
								 stride = 2)
		self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
		self.fc = nn.Linear(512 * block.expansion, num_classes)

	def _make_layer(self, block, out_channels, num_blocks, stride):
		strides = [stride] + [1] * (num_blocks - 1)
		layers = []
		for stride in strides:
			layers.append(block(self.in_channels, out_channels, stride))
			self.in_channels = out_channels * block.expansion
		return nn.Sequential(*layers)

	def forward(self, x):
		out = self.conv1(x)
		out = self.bn1(out)
		out = self.relu(out)
		out = self.layer1(out)
		out = self.layer2(out)
		out = self.layer3(out)
		out = self.layer4(out)
		out = self.avg_pool(out)
		out = out.view(out.size(0), -1)
		out = self.fc(out)
		return out

def ResNet18(num_classes = 10):
	return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

def ResNet34(num_classes = 10):
	return ResNet(BasicBlock, [3, 4, 6, 3], num_classes)

resnet18_model = ResNet18()
resnet_params = sum(p.numel() for p in resnet18_model.parameters())
print(resnet_params)

test_input = torch.randn(1, 3, 32, 32)
resnet_output = resnet18_model(test_input)
print(resnet_output.shape)

11173962
torch.Size([1, 10])


In [15]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels = 3,
                               out_channels = 16,
                               kernel_size = 3,
                               stride = 1,
                               padding = 1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size = 2,
                                  stride = 2)
        self.conv2 = nn.Conv2d(in_channels = 16,
                               out_channels = 32,
                               kernel_size = 3,
                               stride = 1,
                               padding = 1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size = 2,
                                  stride = 2)
        self.fc = nn.Linear(2048, 10)

    def forward(self, x):
        # 첫번째 블록
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        # 두번째 블록
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        # 평탄화
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [23]:
#Scheduler
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau, CosineAnnealingLR
import torch.optim as optim

demo_model = SimpleCNN()
demo_optimizer = optim.SGD(demo_model.parameters(),
                           lr = 0.1,
                           momentum = 0.9,
                           weight_decay = 5e-4)
step_scheduler = StepLR(demo_optimizer,
                        step_size = 30,
                        gamma = 0.1)
plateau_scheduler = ReduceLROnPlateau(demo_optimizer,
                                      mode = 'min',
                                      factor = 0.1,
                                      patience = 10)
cosine_scheduler = CosineAnnealingLR(demo_optimizer,
                                     T_max = 200)

def initialize_weights(model, init_type = 'kaiming'):
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            if init_type == 'kaiming':
                nn.init.kaiming_normal_(m.weight,
                                        mode = 'fan_out',
                                        nonlinearity = 'relu')
            elif init_type == 'xavier':
                nn.init.xavier_normal_(m.weight)
            elif init_type == 'normal':
                nn.init.normal_(m.weight,
                                mean = 0,
                                std = 0.01)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Linear):
            if init_type == 'kaiming':
                nn.init.kaiming_normal_(m.weight,
                                        mode = 'fan_out',
                                        nonlinearity = 'relu')
            elif init_type == 'xavier':
                nn.init.xavier_normal_(m.weight)
            elif init_type == 'normal':
                nn.init.normal_(m.weight,
                                mean = 0,
                                std = 0.01)
                nn.init.constant_(m.bias, 0)

initialize_weights(demo_model, 'kaiming')